<img width="20%" alt="EarthDaily Analytics" src="https://raw.githubusercontent.com/earthdaily/Images/main/Corporate/EarthDaily.png" style="border-radius: 15%">

# 📘 EDAgro - Emergence Processor Bulk Extraction

## **✅ Step 1: Initialisation**

In [ ]:
# Bootstrap: ensure src/ is on sys.path for edagro_utils imports
import sys
from pathlib import Path

_src = str(Path().resolve().parent / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from edagro_utils.notebook_setup import init
init()

In [1]:
from edagro_utils.services.workflow_manager import WorkflowManager
import os
manager = WorkflowManager("prod")

c:\Users\vl\Documents\Github\ApiShowcase-\analytic_extraction_base\src\utils\geometry.py:17: FutureWarning: WKTReadingError is deprecated and will be removed in a future version. Use ShapelyError instead (functions previously raising {name} will now raise a ShapelyError instead).
  from shapely.errors import WKTReadingError
c:\Users\vl\Documents\Github\ApiShowcase-\analytic_extraction_base\src\utils\workflow_manager.py:14: FutureWarning: WKTReadingError is deprecated and will be removed in a future version. Use ShapelyError instead (functions previously raising {name} will now raise a ShapelyError instead).
  from shapely.errors import WKTReadingError
2026-03-08 09:30:27 | INFO     | edagro_utils.core.logging_setup:setup_logging | ✅ Logging system initialized
2026-03-08 09:30:27 | INFO     | edagro_utils.core.logging_setup:setup_logging | 📁 Log directory: c:\Users\vl\Documents\Github\ApiShowcase-\analytic_extraction_base\src\logs
2026-03-08 09:30:27 | INFO     | edagro_utils.services.w

👤 User: KA_Team_Grower
🔑 New token acquired for env=prod, valid until 2026-03-08 10:30:28.159033


2026-03-08 09:30:29 | WARNING  | edagro_utils.core.identity:initialize_s3_client | ⚠️ S3 client initialized but connection test failed: An error occurred (AccessDenied) when calling the ListBuckets operation: User: arn:aws:iam::546530222453:user/sa-key-account is not authorized to perform: s3:ListAllMyBuckets because no identity-based policy allows the s3:ListAllMyBuckets action
2026-03-08 09:30:29 | SUCCESS  | edagro_utils.services.workflow_manager:__init__ | ✅ WorkflowManager initialized
2026-03-08 09:30:29 | INFO     | edagro_utils.services.workflow_manager:__init__ |    Environment: prod
2026-03-08 09:30:29 | INFO     | edagro_utils.services.workflow_manager:__init__ |    Results dir: results
2026-03-08 09:30:29 | INFO     | edagro_utils.services.workflow_manager:__init__ |    Partials dir: partials


## **🛠️ Step 2: Get entities**

### Option 1 - Load entities from Earthdaily platform

In [2]:
manager.load_seasonfields()

print("\n🔎 First entity loaded:")
print(manager.sfd_list[['id', 'name']].head())

2026-03-08 09:30:29 | INFO     | edagro_utils.core.base_extractor:_setup_logger | 🔗 EntityManager initialized within workflow
2026-03-08 09:30:29 | INFO     | edagro_utils.extractors.Entity_management:__init__ | 🏗️ EntityManager initialized for env: prod
2026-03-08 09:30:29 | INFO     | edagro_utils.extractors.Entity_management:get_seasonfields | Fetching seasonfields (field_id=None, crop=None, sowing_gte=None, sowing_lte=None, farm_name=None)
2026-03-08 09:30:35 | INFO     | edagro_utils.extractors.Entity_management:get_seasonfields | Retrieved 19553 seasonfields


✅ Loaded 19553 seasonfields

🔎 First entity loaded:
        id  name
0  z361x33  None
1  7e5gwem  None
2  2apvwaw  None
3  3aj1ma7  None
4  w36vx35  None


### Option 2 - Load entities from Earthdaily platform (Batch method - Recommended for large datasets)

In [ ]:
# This method uses batch processing to efficiently load 5000 entities

# Load all available entities using batch processing
manager.load_seasonfields_batch()

print("\n🔎 First entity loaded:")
print(manager.sfd_list[['id', 'name']].head())

### Option 3 -Load entities from file

In [ ]:
file = "test.shp"
input_path= os.getcwd()+"\\inputs\\"
file_path = os.path.join(input_path,file)

In [ ]:
manager.load_sfd_list(file_path)
print(manager.sfd_list.head())

## **📥 Step 3: Extract analytics - Debug function from processor_emergence_functions.py**

### 🗺️ Configure extraction

In [5]:
# Import your class
from edagro_utils.processors.processor_emergence_functions import EmergenceExtractor
extractor = EmergenceExtractor(manager.bearer_token, manager.token_expiration,config=manager.config )

# Define column mapping to match DataFrame column names from the platform
column_mapping = {"crop": "crop.id"}

extractor.setup_emergence_parameters(
                                    emergence_type="INSEASON",
                                    season_duration=120,
                                    season_start_day=1,
                                    season_start_month=4,
                                    year='2025',
                                    data_source='LR',
                                    publish_af= True,
                                    partial_frequency=50,
                                    keep_geometry=False,
                                    column_mapping=column_mapping
                                    )

2026-03-08 09:32:36 | INFO     | edagro_utils.core.base_extractor:_setup_logger | 🚀 EmergenceExtractor initialized (standalone mode)
2026-03-08 09:32:36 | INFO     | edagro_utils.processors.processor_emergence_functions:__init__ | 🌱 EmergenceExtractor ready for prod environment
2026-03-08 09:32:36 | INFO     | edagro_utils.processors.processor_emergence_functions:setup_emergence_parameters | Configuring emergence parameters...
2026-03-08 09:32:36 | INFO     | edagro_utils.core.base_extractor:set_column_mapping | Column mapping updated: {'crop': 'crop.id'}
2026-03-08 09:32:36 | SUCCESS  | edagro_utils.processors.processor_emergence_functions:setup_emergence_parameters | ✅ Emergence parameters configured successfully


🌍 Emergence parameters configured:
   emergence_type: INSEASON
   season_duration: 120
   season_start_month: 4
   season_start_day: 1
   year: 2025
   data_source: LR
   publish_af: True
   partial_frequency: 50
   keep_geometry: False


### 🗺️ Test functions

In [6]:
# Prepare test seasonfield_data
seasonfield_data = {
    "id": "z361x33",
    "geometry": "POLYGON ((-58.94540508 -13.72028589, -58.942163 -13.73172321, -58.928124090000004 -13.730314100000001, -58.93159922 -13.71888551, -58.94540508 -13.72028589))",
    "crop": "CORN"
}

#### Test get_emergence_api

In [7]:
print("\n--- Test: get_emergence ---")
# Invoke the function
try:
    result = extractor.get_emergence_api(seasonfield_data)
    print("✅ Raw API response received:")
    print(result if isinstance(result, dict) else result[:500])
except Exception as e:
    print(f"❌ API call failed: {e}")
    


2026-03-08 09:32:46 | INFO     | edagro_utils.processors.processor_emergence_functions:get_emergence_api | Requesting emergence data for entity z361x33



--- Test: get_emergence ---


2026-03-08 09:32:53 | SUCCESS  | edagro_utils.processors.processor_emergence_functions:get_emergence_api | ✅ Emergence data retrieved for entity z361x33


✅ Raw API response received:
{'id': 'z361x33', 'data': {'EmergenceDate': '0001-01-01', 'EmergenceStatus': False, 'ConfirmationStatus': 'no_emergence'}, 'statusUpdated': True}


#### Test get_inseason_monitoring_api_safe

In [8]:
print("\n--- Test: get_emergence_safe ---")
safe_result = extractor.get_emergence_api_safe(seasonfield_data)
print(safe_result)

2026-03-08 09:32:53 | INFO     | edagro_utils.processors.processor_emergence_functions:get_emergence_api | Requesting emergence data for entity z361x33



--- Test: get_emergence_safe ---


2026-03-08 09:32:54 | SUCCESS  | edagro_utils.processors.processor_emergence_functions:get_emergence_api | ✅ Emergence data retrieved for entity z361x33


{'success': True, 'data': {'id': 'z361x33', 'data': {'EmergenceDate': '0001-01-01', 'EmergenceStatus': False, 'ConfirmationStatus': 'no_emergence'}, 'statusUpdated': True}, 'error': None, 'seasonfield_id': 'z361x33'}


In [9]:
print("\n--- Test: format_emergence_json ---")
if safe_result["success"] and safe_result["data"]:
    formatted_df = extractor.format_emergence_json(safe_result["data"])
    print(f"✅ Formatted DataFrame with {len(formatted_df)} rows:")
    display(formatted_df.head())
else:
    print("⚠️ Skipping format_emergence_json: No valid data from API.")


--- Test: format_emergence_json ---
✅ Formatted DataFrame with 1 rows:


,entity_id,emergence_date,emergence_status,confirmation_status
0,z361x33,None,False,no_emergence


#### Test format_inseason_json

### 🗺️ process_single_entity_emergence

In [10]:
import pandas as pd
row = pd.Series({
    "id": "z361x33",
    "geometry": "POLYGON ((-58.94540508 -13.72028589, -58.942163 -13.73172321, -58.928124090000004 -13.730314100000001, -58.93159922 -13.71888551, -58.94540508 -13.72028589))",
    "crop":"OTHERS"
})

result = extractor.process_single_entity_emergence(row)

print(result)

2026-03-08 09:33:00 | INFO     | edagro_utils.processors.processor_emergence_functions:process_single_entity_emergence | Calling emergence API for entity z361x33 (with retry logic)
2026-03-08 09:33:00 | INFO     | edagro_utils.processors.processor_emergence_functions:get_emergence_api | Requesting emergence data for entity z361x33
2026-03-08 09:33:02 | SUCCESS  | edagro_utils.processors.processor_emergence_functions:get_emergence_api | ✅ Emergence data retrieved for entity z361x33
2026-03-08 09:33:02 | SUCCESS  | edagro_utils.processors.processor_emergence_functions:process_single_entity_emergence | ✅ Successfully processed entity z361x33


{'data':   entity_id emergence_date  emergence_status confirmation_status       id  \
0   z361x33           None             False        no_emergence  z361x33   

     crop  
0  OTHERS  , 'error': None}


### 🗺️ process_emergence_bulk_extraction_parallel

In [11]:

top25 = manager.sfd_list.head(30)
print(top25.columns)
# Launch extraction with 10 threads 
result = extractor.process_emergence_bulk_extraction_parallel(
    entity_list=top25,
    params=None,
    max_workers=20,
    output_path=manager.output_result_dir,
    fail_safe=False,
    filter_column="crop.id",
    filter_value="CORN",
    filter_type="exclude", # filter type used to 'include' or 'exclude' row matching column and value filter
    merge_existing='auto',
    skip_export=False,
    prefix="emergence"
)

print(f"\nResults summary:")
print(f"Total: {result['total_calculations']}")
print(f"Success: {result['successful_calculations']}")
print(f"Failed: {result['failed_calculations']}")
print(result["results_df"].columns)
print("\n🔍 First 3 errors:")
for i, error in enumerate(result['global_errors'][:3]):
    print(f"\nError {i+1}:")
    for key, value in error.items():
        print(f"  {key}: {value}")

2026-03-08 09:33:04 | INFO     | edagro_utils.processors.processor_emergence_functions:process_emergence_bulk_extraction_parallel | ============================================================
2026-03-08 09:33:04 | INFO     | edagro_utils.processors.processor_emergence_functions:process_emergence_bulk_extraction_parallel | Starting bulk emergence extraction: emergence
2026-03-08 09:33:04 | INFO     | edagro_utils.processors.processor_emergence_functions:process_emergence_bulk_extraction_parallel | ============================================================
2026-03-08 09:33:04 | INFO     | edagro_utils.processors.processor_emergence_functions:process_emergence_bulk_extraction_parallel | Total input entities: 30
2026-03-08 09:33:04 | INFO     | edagro_utils.processors.processor_emergence_functions:process_emergence_bulk_extraction_parallel | Max workers: 20
2026-03-08 09:33:04 | INFO     | edagro_utils.processors.processor_emergence_functions:process_emergence_bulk_extraction_parallel |

Index(['id', 'name', 'geometry', 'sowingDate', 'crop.code', 'crop.id',
       'field.id', 'field.farm.grower.companyName',
       'field.farm.grower.firstname'],
      dtype='object')
🔄 Processing Emergence for 30 entities in parallel...


2026-03-08 09:33:04 | INFO     | edagro_utils.processors.processor_emergence_functions:process_single_entity_emergence | Calling emergence API for entity ajladjv (with retry logic)
2026-03-08 09:33:04 | INFO     | edagro_utils.processors.processor_emergence_functions:get_emergence_api | Requesting emergence data for entity ajladjv
2026-03-08 09:33:04 | INFO     | edagro_utils.processors.processor_emergence_functions:process_single_entity_emergence | Calling emergence API for entity 5nr1vn9 (with retry logic)
🌿 Processing Emergence:   0%|          | 0/30 [00:00<?, ?entity/s]2026-03-08 09:33:04 | INFO     | edagro_utils.processors.processor_emergence_functions:get_emergence_api | Requesting emergence data for entity 7e5gwem
2026-03-08 09:33:04 | INFO     | edagro_utils.processors.processor_emergence_functions:process_single_entity_emergence | Calling emergence API for entity y3wnx36 (with retry logic)
2026-03-08 09:33:04 | INFO     | edagro_utils.processors.processor_emergence_functions:


⏱️ Total processing time: 12.63 seconds
✅ Successful calculations: 30/30
📋 No skipped entities to merge
📦 Final export to: results
📄 Results saved to: results\emergence_results_20260308_093317_final.csv

Results summary:
Total: 30
Success: 30
Failed: 0
Index(['entity_id', 'emergence_date', 'emergence_status',
       'confirmation_status', 'id', 'name', 'sowingDate', 'crop.code',
       'crop.id', 'field.id', 'field.farm.grower.companyName',
       'field.farm.grower.firstname'],
      dtype='object')

🔍 First 3 errors:


In [10]:
print(result["results_df"])

   entity_id emergence_date  emergence_status confirmation_status       id  \
0    z361x33           None             False        no_emergence  z361x33   
1    2apvwaw           None             False        no_emergence  2apvwaw   
2    jldjv5d           None             False        no_emergence  jldjv5d   
3    r96b3ed           None             False        no_emergence  r96b3ed   
4    1aqpwma           None             False        no_emergence  1aqpwma   
5    bgevyaw           None             False        no_emergence  bgevyaw   
6    7e5gwem           None             False        no_emergence  7e5gwem   
7    bgevya7           None             False        no_emergence  bgevya7   
8    e9jy3xn     2025-04-03              True           confirmed  e9jy3xn   
9    e9jy3xp           None             False        no_emergence  e9jy3xp   
10   w36vx35           None             False        no_emergence  w36vx35   
11   5nr1vn9           None             False        no_emergenc

In [11]:
# Get the clean DataFrame
results=result["results_df"]
print(results.columns)

Index(['entity_id', 'emergence_date', 'emergence_status',
       'confirmation_status', 'id', 'name', 'sowingDate', 'crop.code',
       'crop.id', 'field.id', 'field.farm.grower.companyName',
       'field.farm.grower.firstname'],
      dtype='object')
